## Data Gathering, Removing Outliers, Training and Evaluation

In [31]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
import shap

os.makedirs('image', exist_ok=True)
os.makedirs('CSV', exist_ok=True)


metrics = pd.read_csv('./CSV/creator_metrics_final.csv')
videos = pd.read_csv('./CSV/creator_videos_cleaned.csv')
profiles = pd.read_csv('./CSV/creator_profiles_cleaned.csv')

video_aggs = videos.groupby('creator_id').agg({
    'view_count': 'mean',
    'like_count': 'mean',
    'comment_count': 'mean',
    'share_count': 'mean',
}).reset_index()
video_aggs.columns = ['creator_id', 'avg_view_count', 'avg_like_count', 'avg_comment_count', 'avg_share_count']

df = metrics.merge(profiles, on='creator_id', how='left')
df = df.merge(video_aggs, on='creator_id', how='left')
print(df.columns)

target = 'avg_comment_to_like_ratio'


features = [
    'avg_hook_retention_rate', 'action_theme_diversity_score',
    'price_to_median_views_ratio', 'trend_momentum',
    'negative_sentiment_rate', 'authenticity', 'comment_sentiment',
    'hashtag_strategy', 'past_collaborations_success',
    'price_to_engagement_ratio', 'daily_profile_views',
    'content_diversity_score', 'strongest_engagement_growth_post_upload',
    'brand_risk_score', 'video_quality', 'engagement_velocity',
    'trend_resonance', 'hook_retention_rate', 'upload_consistency',
    'trend_personalization', 'avg_view_count', 'avg_like_count',
    'avg_comment_count', 'avg_share_count'
]

model_data = df[["creator_id"] + features + [target]].dropna().copy()
z = np.abs((model_data[features + [target]] - model_data[features + [target]].mean()) /
           (model_data[features + [target]].std() + 1e-8))
model_data = model_data[(z < 3).all(axis=1)]


unique_creators = model_data['creator_id'].unique()
np.random.seed(42)
np.random.shuffle(unique_creators)
split = int(len(unique_creators) * 0.85)
train_creator_ids = set(unique_creators[:split])
test_creator_ids = set(unique_creators[split:])

train_mask = model_data['creator_id'].isin(train_creator_ids)
test_mask = model_data['creator_id'].isin(test_creator_ids)

X_train = model_data.loc[train_mask, features]
y_train = model_data.loc[train_mask, target]
creator_id_train = model_data.loc[train_mask, 'creator_id']

X_test = model_data.loc[test_mask, features]
y_test = model_data.loc[test_mask, target]
creator_id_test = model_data.loc[test_mask, 'creator_id']

print(f"Unique creators in train: {len(set(creator_id_train))}, test: {len(set(creator_id_test))}")
print(f"Overlap: {len(set(creator_id_train) & set(creator_id_test))}")


plt.figure(figsize=(9, 4))
plt.hist(y_train, bins=40, alpha=0.7, label='Train', color='royalblue')
plt.hist(y_test, bins=40, alpha=0.6, label='Test', color='orange')
plt.legend()
plt.title('Target Distribution: avg_comment_to_like_ratio')
plt.xlabel('Value')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig("image/target_distribution_train_vs_test.png")
plt.close()


mean_pred = np.mean(y_train)
rmse_baseline = np.sqrt(mean_squared_error(y_test, [mean_pred]*len(y_test)))
r2_baseline = r2_score(y_test, [mean_pred]*len(y_test))
print(f"Baseline (mean predictor) - Test RMSE: {rmse_baseline:.3f}, R²: {r2_baseline:.3f}")


scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


model = XGBRegressor(
    n_estimators=150, max_depth=4, learning_rate=0.08,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.2,
    random_state=42, eval_metric='rmse'
)
model.fit(X_train_scaled, y_train, eval_set=[(X_train_scaled, y_train), (X_test_scaled, y_test)], verbose=False)


y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train R²: {train_r2:.3f}, Test R²: {test_r2:.3f}")
print(f"Train RMSE: {train_rmse:.3f}, Test RMSE: {test_rmse:.3f}")


train_results = pd.DataFrame({
    'creator_id': creator_id_train,
    'groundtruth': y_train,
    'prediction': y_train_pred
}, index=y_train.index)
test_results = pd.DataFrame({
    'creator_id': creator_id_test,
    'groundtruth': y_test,
    'prediction': y_test_pred
}, index=y_test.index)
train_results.to_csv('./CSV/train_predictions_vs_groundtruth.csv', index=False)
test_results.to_csv('./CSV/test_predictions_vs_groundtruth.csv', index=False)


results = model.evals_result()
epochs = range(1, len(results['validation_0']['rmse']) + 1)
plt.figure(figsize=(7, 5))
plt.plot(epochs, results['validation_0']['rmse'], label='Train RMSE')
plt.plot(epochs, results['validation_1']['rmse'], label='Test RMSE')
plt.xlabel('Boosting Round')
plt.ylabel('RMSE')
plt.title('Learning Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("image/learning_curve.png")
plt.close()


importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(7, 5))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [features[i] for i in indices], rotation=45, ha='right')
plt.title('Feature Importance')
plt.tight_layout()
plt.savefig("image/feature_importance.png")
plt.close()


plt.figure(figsize=(7, 5))
plt.scatter(y_test_pred, y_test - y_test_pred, alpha=0.7)
plt.axhline(0, color='r', ls='--')
plt.xlabel('Predicted')
plt.ylabel('Residuals')
plt.title('Residuals Plot (Test set)')
plt.tight_layout()
plt.savefig("image/residuals_plot.png")
plt.close()

print("Calculated SHAP values...")
explainer = shap.Explainer(model, X_train_scaled)
shap_values = explainer(X_test_scaled[:100])
shap.summary_plot(shap_values, X_test.iloc[:100], feature_names=features, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("image/shap_summary_bar.png")
plt.close()

print("\nAll outputs saved. Review test R² and RMSE, and analyze SHAP and feature importance for new insight.")


Index(['creator_id', 'avg_comment_to_like_ratio', 'avg_hook_retention_rate',
       'action_theme_diversity_score', 'price_to_median_views_ratio',
       'trend_momentum', 'timestamp', 'usage_of_low_quality_products',
       'negative_sentiment_rate', 'authenticity', 'comment_sentiment',
       'hashtag_strategy', 'past_collaborations_success',
       'price_to_engagement_ratio', 'daily_profile_views',
       'content_diversity_score', 'strongest_engagement_growth_post_upload',
       'brand_risk_score', 'follower_count', 'video_quality',
       'engagement_velocity', 'creator_price', 'trend_resonance',
       'hook_retention_rate', 'upload_consistency', 'trend_personalization',
       'avg_view_count', 'avg_like_count', 'avg_comment_count',
       'avg_share_count'],
      dtype='object')
Unique creators in train: 164, test: 29
Overlap: 0
Baseline (mean predictor) - Test RMSE: 0.081, R²: -0.012
Train R²: 1.000, Test R²: -5.355
Train RMSE: 0.002, Test RMSE: 0.203
Calculating SHAP value

/var/folders/h9/c7hnw281781_jhjxkvz6f3mw0000gn/T/ipykernel_63682/2008832023.py:171: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_test.iloc[:100], feature_names=features, plot_type="bar", show=False)


## Rising star

In [28]:
rising_star_threshold = np.percentile(test_results['prediction'], 90)
test_results['rising_star'] = test_results['prediction'] >= rising_star_threshold


test_results.to_csv('./CSV/test_predictions_vs_groundtruth.csv', index=False)

print(f"\nRising star threshold (top 10%): {rising_star_threshold:.4f}")
print(f"Number of rising stars in test set: {test_results['rising_star'].sum()}")
print("\nRising star creators (top 5 shown):")
print(test_results[test_results['rising_star']].head())



Rising star threshold (top 10%): 0.1980
Number of rising stars in test set: 75

Rising star creators (top 5 shown):
                                creator_id  groundtruth  prediction  \
1720  d1bfd78d-fc9b-4377-8a27-5d2038beff87     0.046023    0.684667   
1721  d1bfd78d-fc9b-4377-8a27-5d2038beff87     0.046023    0.684667   
1722  d1bfd78d-fc9b-4377-8a27-5d2038beff87     0.046023    0.684667   
1723  d1bfd78d-fc9b-4377-8a27-5d2038beff87     0.046023    0.684667   
1724  d1bfd78d-fc9b-4377-8a27-5d2038beff87     0.046023    0.684667   

      rising_star  
1720         True  
1721         True  
1722         True  
1723         True  
1724         True  


In [29]:
train_ids = set(pd.read_csv('CSV/train_predictions_vs_groundtruth.csv')['creator_id'])
test_ids = set(pd.read_csv('CSV/test_predictions_vs_groundtruth.csv')['creator_id'])
overlap = train_ids & test_ids
print(f"Number of overlapping creator_ids: {len(overlap)}")

Number of overlapping creator_ids: 0
